# 실습 5주차: 이미지를 텐서로, 콘볼루션을 코드로

> **시나리오 — 오늘 만들 것**
>
>
> 지난 4주 동안 다룬 입력은 `(n, p)` 짜리 **표**였다. 오늘부터 입력은 **사진**이다.
>
> $$\text{사진 한 장} \;\longrightarrow\; (C, H, W) \text{ 숫자 배열} \;\longrightarrow\; \boxed{\text{콘볼루션}} \;\longrightarrow\; \text{분류}$$
>
> 오늘은 **콘볼루션을 이중 for문으로 직접 만들어** `F.conv2d` 와 값이 같은지 확인하고,
> 마지막에 **파라미터 235개짜리 CNN으로 실제 옷 사진을 분류**한다.
> 그리고 학습이 끝난 커널을 꺼내 **모형이 무엇을 배웠는지 눈으로 본다.**
>
> - **대응 이론**: [Ch05 이미지 데이터와 컴퓨터 비전](ch05.qmd), [Ch06 콘볼루션 연산](ch06.qmd)
> - 코드는 완성되어 있다. **직접 해보기** 칸은 스스로 채운 뒤 아래 정답과 맞춰 본다.
> - 데이터: FashionMNIST(흑백 28×28), CIFAR-10(컬러 32×32)


> **이번 주에 익히는 것**
>
>
> | 개념 | 이론과의 대응 |
> |------|------|
> | 이미지 = 숫자 배열 | 픽셀, 채널 (Ch05) |
> | `(H, W)` · `(H, W, C)` · `(C, H, W)` | numpy와 PyTorch의 축 순서 차이 |
> | 리사이즈 · `÷255` · 표준화 | 전처리 파이프라인 (Ch05) |
> | `(N, C, H, W)` | 배치 텐서 |
> | 이중 for문 콘볼루션 → `F.conv2d` | 콘볼루션 연산 (Ch06) |
> | 출력 크기 공식 검산 | $\lfloor (N-K+2P)/S \rfloor + 1$ (Ch06) |
> | 파라미터 수: FC vs Conv | 파라미터 폭발과 가중치 공유 |


---

# 1. 이미지 한 장을 숫자로 열기

이미지는 특별한 자료형이 아니다. **숫자가 격자로 배열된 것**뿐이다.

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision import datasets, transforms

np.set_printoptions(linewidth=140)
torch.manual_seed(42)

train_raw = datasets.FashionMNIST(root='./data', train=True, download=True)
print('데이터 수 :', len(train_raw))
print('클래스    :', train_raw.classes)

## 1-1. 한 장을 꺼내 모양을 본다

In [ ]:
img, label = train_raw[0]          # PIL 이미지 한 장과 정답 번호
print('타입 :', type(img).__name__)
print('크기 :', img.size, '   모드:', img.mode)   # 'L' = 흑백 1채널
print('레이블:', label, '→', train_raw.classes[label])

a = np.array(img)                  # PIL → numpy 배열
print('\nshape :', a.shape)
print('dtype :', a.dtype)
print('범위  :', a.min(), '~', a.max())

> `uint8` 은 0~255 정수를 담는 자료형이다. 흑백 이미지 한 장은 `(높이, 너비)` 모양의
> 2차원 배열이고, 각 칸이 **밝기값 하나**다.


## 1-2. 픽셀 값을 직접 본다

이미지의 가운데 부분을 숫자로 그대로 출력한다.

In [ ]:
patch = a[10:18, 8:20]
print('패치 shape:', patch.shape)
print(patch)

같은 영역을 그림으로 보면 숫자와 밝기가 대응한다는 것이 보인다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.5, 3.4))
axes[0].imshow(a, cmap='gray'); axes[0].set_title(f'full 28x28 ({train_raw.classes[label]})', fontsize=9)
axes[1].imshow(patch, cmap='gray'); axes[1].set_title('patch a[10:18, 8:20]', fontsize=9)
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

## 1-3. 여러 장의 통계

In [ ]:
sub = np.stack([np.array(train_raw[i][0]) for i in range(1000)])
print('1000장 묶은 shape :', sub.shape)      # (N, H, W)
print('전체 평균 밝기    :', round(float(sub.mean()), 2))
print('전체 표준편차     :', round(float(sub.std()), 2))
print('장별 평균 shape   :', sub.mean(axis=(1, 2)).shape)   # 축 두 개를 없앤다

1주차의 **축(axis)** 규칙이 그대로 적용된다 — `axis=(1,2)` 는 높이와 너비를 없애고
**장마다 하나씩** 값을 남긴다.

---

# 2. 컬러 이미지와 채널

## 2-1. `(H, W, C)`

In [ ]:
cifar_raw = datasets.CIFAR10(root='./data', train=True, download=True)
img_c, label_c = cifar_raw[0]
arr = np.array(img_c)

print('클래스   :', cifar_raw.classes)
print('레이블   :', label_c, '→', cifar_raw.classes[label_c])
print('shape    :', arr.shape)          # (높이, 너비, 채널)
print('dtype    :', arr.dtype)
print('R,G,B 평균:', arr.mean(axis=(0, 1)).round(2))

## 2-2. 채널을 분해해서 본다

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(11, 3))
axes[0].imshow(arr); axes[0].set_title(f'RGB ({cifar_raw.classes[label_c]})', fontsize=9)
for i, (name, cm) in enumerate(zip(['R', 'G', 'B'], ['Reds', 'Greens', 'Blues'])):
    axes[i+1].imshow(arr[:, :, i], cmap=cm, vmin=0, vmax=255)
    axes[i+1].set_title(f'channel {i} ({name})', fontsize=9)
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

`arr[:, :, 0]` 은 **마지막 축에서 0번째**를 꺼낸 것 — 빨강 채널 한 장이다.

## 2-3. `(H, W, C)` ↔ `(C, H, W)`

In [ ]:
chw = arr.transpose(2, 0, 1)      # 축 순서를 (2,0,1)로 재배치
print('numpy  HWC :', arr.shape)
print('numpy  CHW :', chw.shape)

t_hwc = torch.tensor(arr)
t_chw = t_hwc.permute(2, 0, 1)    # PyTorch에서는 permute
print('torch  HWC :', tuple(t_hwc.shape))
print('torch  CHW :', tuple(t_chw.shape))
print('값이 같은가:', np.array_equal(t_chw.numpy(), chw))

> **축 순서는 라이브러리마다 다르다**
>
>
> | | 축 순서 | 쓰는 곳 |
> |---|---|---|
> | PIL · matplotlib · OpenCV | `(H, W, C)` | 이미지를 **보여줄 때** |
> | PyTorch | `(C, H, W)` | 신경망에 **넣을 때** |
>
> `plt.imshow()` 에 `(3, 32, 32)` 텐서를 그대로 넣으면 에러가 난다.
> 보여줄 때는 `.permute(1, 2, 0)` 으로 되돌린다.

In [ ]:
plt.figure(figsize=(2.4, 2.4))
plt.imshow(t_chw.permute(1, 2, 0).numpy())   # 다시 HWC로 되돌려서 표시
plt.axis('off'); plt.tight_layout(); plt.show()

> **직접 해보기 ① — 축 순서 되돌리기**
>
>
> `(3, 32, 32)` 짜리 텐서 `t_chw` 를 **numpy 함수로** `(32, 32, 3)` 으로 되돌리시오.
> (힌트: `.numpy()` 뒤에 `transpose`)

In [ ]:
# ✏️ 직접 채워 보세요
back = None            # ← 여기를 채우세요

assert back is not None and back.shape == (32, 32, 3), '모양을 확인하세요'
assert np.array_equal(back, arr), '값이 원본과 달라졌습니다'
print('통과', back.shape)

In [ ]:
# ↓ 정답 코드 (먼저 스스로 해 본 뒤에 실행하세요)
back = t_chw.numpy().transpose(1, 2, 0)
assert back.shape == (32, 32, 3) and np.array_equal(back, arr)
print('통과', back.shape)

---

# 3. 신경망에 넣기 전 — 리사이즈와 정규화

Ch05에서 두 가지 문제를 봤다. **크기가 제각각**인 것과 **값의 범위가 0~255**인 것이다.

## 3-1. 리사이즈

In [ ]:
resize = transforms.Resize((64, 64))
small = transforms.Resize((8, 8))
print('원본     :', img_c.size)
print('64x64로  :', resize(img_c).size)
print('8x8로    :', small(img_c).size)

fig, axes = plt.subplots(1, 3, figsize=(8, 3))
for ax, im, t in zip(axes, [resize(img_c), img_c, small(img_c)], ['64x64', '32x32 (original)', '8x8']):
    ax.imshow(np.array(im)); ax.set_title(t, fontsize=9); ax.axis('off')
plt.tight_layout(); plt.show()

키운다고 정보가 늘지는 않는다. 줄이면 정보는 **돌이킬 수 없이 사라진다**.

## 3-2. `ToTensor` 가 하는 두 가지

In [ ]:
to_tensor = transforms.ToTensor()
x = to_tensor(img_c)

print('shape :', tuple(x.shape))     # CHW로 바뀌었다
print('dtype :', x.dtype)            # float32로 바뀌었다
print('범위  :', float(x.min()), '~', float(x.max()))

두 가지가 한꺼번에 일어났다 — **`(H,W,C)` → `(C,H,W)`** 와 **`÷255`**.
직접 계산해서 대조한다.

In [ ]:
manual = arr.transpose(2, 0, 1).astype(np.float32) / 255.0
print('직접 계산과 일치:', np.allclose(x.numpy(), manual))
print('첫 픽셀 (R,G,B):', arr[0, 0], '→', x[:, 0, 0].numpy().round(6))

## 3-3. 표준화 (`Normalize`)

$$x' = \frac{x - \mu}{\sigma}$$

In [ ]:
mean = (0.5, 0.5, 0.5)
std  = (0.5, 0.5, 0.5)
normalize = transforms.Normalize(mean, std)
xn = normalize(x)

print('정규화 전 범위:', round(float(x.min()), 4), '~', round(float(x.max()), 4))
print('정규화 후 범위:', round(float(xn.min()), 4), '~', round(float(xn.max()), 4))

# 채널 0의 첫 픽셀로 직접 검산
print('\n직접 계산:', round((float(x[0,0,0]) - 0.5) / 0.5, 6))
print('Normalize:', round(float(xn[0,0,0]), 6))

`mean=0.5, std=0.5` 는 `[0,1]` 범위를 `[-1,1]` 로 옮긴다. 실제 데이터의 평균·표준편차를
쓰면 각 채널이 평균 0, 표준편차 1 근처가 된다.

In [ ]:
# 훈련 데이터 1000장으로 채널별 통계를 구한다 (4주차 규칙: 통계는 훈련 세트에서만)
sub_c = np.stack([np.array(cifar_raw[i][0]) for i in range(1000)]).astype(np.float32) / 255.0
ch_mean = sub_c.mean(axis=(0, 1, 2))    # 장·높이·너비를 없애고 채널만 남긴다
ch_std  = sub_c.std(axis=(0, 1, 2))
print('채널별 평균  :', ch_mean.round(4))
print('채널별 표준편차:', ch_std.round(4))

## 3-4. 파이프라인으로 묶기

In [ ]:
pipeline = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),                          # CHW + ÷255
    transforms.Normalize(tuple(ch_mean), tuple(ch_std)),
])

out = pipeline(img_c)
print('shape :', tuple(out.shape))
print('채널별 평균:', out.mean(dim=(1, 2)).numpy().round(4))

> **`Normalize` 는 `ToTensor` **뒤에** 와야 한다**
>
>
> `ToTensor` 는 PIL 이미지를 받고, `Normalize` 는 텐서를 받는다. 순서를 바꾸면 에러가 난다.


---

# 4. 배치 텐서 `(N, C, H, W)`

신경망은 한 장씩이 아니라 **여러 장을 한 번에** 처리한다.

In [ ]:
from torch.utils.data import DataLoader

train_ds = datasets.CIFAR10(root='./data', train=True, download=False, transform=pipeline)
loader = DataLoader(train_ds, batch_size=64, shuffle=True)

xb, yb = next(iter(loader))
print('입력 배치 shape :', tuple(xb.shape))    # (N, C, H, W)
print('정답 배치 shape :', tuple(yb.shape))
print('N=배치 크기, C=채널, H=높이, W=너비')

| 기호 | 의미 | 이 배치에서 |
|:---:|---|:---:|
| N | 한 번에 처리하는 이미지 수 | 64 |
| C | 채널 수 | 3 |
| H, W | 높이, 너비 | 32, 32 |

앞으로 나오는 모든 이미지 텐서는 이 네 축을 갖는다.

---

# 5. FC로 이미지를 처리하면 — 파라미터 폭발

Ch05의 손계산을 코드로 확인한다. 이미지를 한 줄로 펴서 완전연결층에 넣어 본다.

In [ ]:
flat = xb.flatten(start_dim=1)     # (N, C, H, W) → (N, C*H*W)
print('펴기 전:', tuple(xb.shape))
print('펴기 후:', tuple(flat.shape), '  =  3 x 32 x 32 =', 3*32*32)

fc = torch.nn.Linear(3*32*32, 128)
n_fc = sum(p.numel() for p in fc.parameters())
print('\nFC 층 파라미터 :', f'{n_fc:,}')
print('공식 3072x128+128:', f'{3*32*32*128 + 128:,}')

이미지 크기를 키우면 어떻게 되는지 표로 만든다.

In [ ]:
import pandas as pd
rows = []
for size in [32, 64, 128, 224]:
    d = 3 * size * size
    rows.append({'이미지': f'{size}x{size}x3', '펼친 길이': f'{d:,}',
                 '은닉 128개일 때 파라미터': f'{d*128 + 128:,}'})
print(pd.DataFrame(rows).to_string(index=False))

224×224 컬러 이미지 하나를 은닉 128개짜리 층에 넣는 것만으로 **1900만 개**가 넘는다.
이것이 Ch05가 말한 파라미터 폭발이다.

---

# 6. 콘볼루션을 직접 구현하기

## 6-1. 이중 for문으로

커널을 한 칸씩 옮기며 **곱해서 더하는** 것이 전부다.

In [ ]:
def conv2d_naive(x, k, b=0.0, stride=1, pad=0):
    """x: (C, H, W), k: (C, KH, KW) → (OH, OW)"""
    if pad > 0:
        x = np.pad(x, ((0, 0), (pad, pad), (pad, pad)))
    C, H, W = x.shape
    _, KH, KW = k.shape
    OH = (H - KH) // stride + 1
    OW = (W - KW) // stride + 1
    out = np.zeros((OH, OW), dtype=np.float32)
    for i in range(OH):
        for j in range(OW):
            patch = x[:, i*stride:i*stride+KH, j*stride:j*stride+KW]
            out[i, j] = (patch * k).sum() + b       # 모든 채널을 한꺼번에 더한다
    return out

## 6-2. 에지 검출 — Ch06의 손계산 재현

왼쪽 절반이 밝고(10) 오른쪽 절반이 어두운(0) 이미지에 세로 에지 커널을 적용한다.

In [ ]:
img_edge = np.zeros((1, 5, 5), dtype=np.float32)
img_edge[0, :, :2] = 10

k_edge = np.array([[[1, 0, -1],
                    [1, 0, -1],
                    [1, 0, -1]]], dtype=np.float32)

print('입력:\n', img_edge[0])
print('\n특성 맵:\n', conv2d_naive(img_edge, k_edge))

값이 큰 곳(30)이 **밝음→어두움 경계**의 위치다. 균일한 오른쪽 영역에서는 0이 나온다.

> **직접 해보기 ② — 가로 에지 커널**
>
>
> 같은 함수로 **가로 방향 에지**를 찾는 커널을 만들어 적용하시오.
> 위쪽이 밝고 아래쪽이 어두운 이미지를 만들어 시험해 보면 된다.

In [ ]:
# ✏️ 직접 채워 보세요
img_h = np.zeros((1, 5, 5), dtype=np.float32)
img_h[0, :2, :] = 10                    # 위쪽 두 줄만 밝게

k_h = None                              # ← 가로 에지 커널 (1, 3, 3)

fm = conv2d_naive(img_h, k_h)
assert k_h is not None and fm.shape == (3, 3), '모양을 확인하세요'
print(fm)

In [ ]:
# ↓ 정답 코드 (먼저 스스로 해 본 뒤에 실행하세요)
img_h = np.zeros((1, 5, 5), dtype=np.float32)
img_h[0, :2, :] = 10

k_h = np.array([[[ 1,  1,  1],
                 [ 0,  0,  0],
                 [-1, -1, -1]]], dtype=np.float32)

print('입력:\n', img_h[0])
print('\n특성 맵:\n', conv2d_naive(img_h, k_h))

## 6-3. 2채널 + 편향 — 도장 문제 재현

Ch06에서 손으로 계산했던 그 예제다.

In [ ]:
x2 = np.array([[[1, 2, 0], [0, 1, 3], [2, 1, 1]],     # 입력 채널 1
               [[0, 1, 1], [2, 0, 1], [1, 2, 0]]],    # 입력 채널 2
              dtype=np.float32)
k2 = np.array([[[1, 0], [0, 1]],                      # 커널 채널 1
               [[0, 1], [1, 0]]],                     # 커널 채널 2
              dtype=np.float32)
b2 = 1.0

print('입력 shape :', x2.shape, '  커널 shape :', k2.shape)
print('출력 크기 공식: (3-2+0)//1 + 1 =', (3-2+0)//1 + 1)
print('\n특성 맵:\n', conv2d_naive(x2, k2, b2))

손으로 구한 값과 같다. **채널별 결과는 따로 쌓이지 않고 더해진다** — 2채널 입력이
1채널 출력이 되는 이유다.

## 6-4. `F.conv2d` 와 대조

PyTorch는 같은 계산을 `(N, C, H, W)` 배치 단위로 한다.

In [ ]:
xt = torch.tensor(x2).unsqueeze(0)      # (C,H,W) → (1,C,H,W)  N축 추가
kt = torch.tensor(k2).unsqueeze(0)      # (C,KH,KW) → (1,C,KH,KW)  출력채널 축 추가
bt = torch.tensor([b2])

print('입력 텐서 :', tuple(xt.shape), '  (N, C_in,  H,  W)')
print('커널 텐서 :', tuple(kt.shape), '  (C_out, C_in, KH, KW)')

y = F.conv2d(xt, kt, bias=bt)
print('출력 텐서 :', tuple(y.shape), '  (N, C_out, OH, OW)')
print('\n값:\n', y.squeeze().numpy())
print('\n직접 구현과 일치:', np.allclose(y.squeeze().numpy(), conv2d_naive(x2, k2, b2)))

> **커널 텐서의 네 축**
>
>
> $$\text{커널} = (C_{\text{out}},\ C_{\text{in}},\ K_H,\ K_W)$$
>
> - $C_{\text{in}}$ 은 **입력 채널 수와 반드시 같아야** 한다.
> - $C_{\text{out}}$ 은 **커널의 개수**이고, 그대로 출력 채널 수가 된다.
>
> 콘볼루션에서 나는 에러의 대부분이 이 두 줄로 설명된다.


---

# 7. 출력 크기 공식 검산

$$\text{출력 크기} = \left\lfloor \frac{N - K + 2P}{S} \right\rfloor + 1$$

공식을 함수로 만들고, `nn.Conv2d` 의 실제 출력과 대조한다.

In [ ]:
def out_size(N, K, P, S):
    return (N - K + 2*P) // S + 1      # 파이썬 // 가 곧 버림이다

cases = [(7, 3, 0, 1), (32, 5, 2, 2), (28, 3, 1, 1), (64, 7, 3, 2), (32, 3, 1, 2)]
rows = []
for N, K, P, S in cases:
    formula = out_size(N, K, P, S)
    conv = torch.nn.Conv2d(1, 1, K, stride=S, padding=P)
    real = conv(torch.zeros(1, 1, N, N)).shape[-1]
    rows.append({'N': N, 'K': K, 'P': P, 'S': S,
                 '공식': formula, 'Conv2d': real, '일치': formula == real})
print(pd.DataFrame(rows).to_string(index=False))

> 두 번째 줄이 시험에 가장 많이 나오는 형태다. $(32-5+4)/2 = 15.5$ 를 **반올림하면 16**,
> 버리면 15이고 여기에 1을 더해 **16**이다. 답은 같지만 과정이 다르다 —
> $(32-5+4)/2 = 15.5 \to 15 \to 16$ 이 올바른 순서다.


## 7-1. 패딩으로 크기를 유지하기

In [ ]:
x_test = torch.zeros(1, 3, 32, 32)
for K in [3, 5, 7]:
    P = (K - 1) // 2                     # same 패딩
    y_same = torch.nn.Conv2d(3, 8, K, padding=P)(x_test)
    y_valid = torch.nn.Conv2d(3, 8, K, padding=0)(x_test)
    print(f'K={K}  P={P} → {tuple(y_same.shape[-2:])} (same)   '
          f'P=0 → {tuple(y_valid.shape[-2:])} (valid)')

$P = (K-1)/2$ 를 쓰면 입력과 출력 크기가 같아진다. 커널 크기가 **홀수**여야
이 값이 정수가 된다.

---

# 8. 커널을 실제 이미지에 적용하기

사람이 설계한 커널 네 개를 실제 사진에 적용해서, 커널마다 다른 것을 본다는 것을 확인한다.

In [ ]:
photo = torch.tensor(np.array(train_raw[9][0]), dtype=torch.float32)[None, None] / 255.0
print('입력 shape:', tuple(photo.shape), ' 클래스:', train_raw.classes[train_raw[9][1]])

kernels = {
    'vertical edge':   [[1, 0, -1], [1, 0, -1], [1, 0, -1]],
    'horizontal edge': [[1, 1, 1], [0, 0, 0], [-1, -1, -1]],
    'blur':            [[1/9]*3]*3,
    'sharpen':         [[0, -1, 0], [-1, 5, -1], [0, -1, 0]],
}

fig, axes = plt.subplots(1, 5, figsize=(13, 3))
axes[0].imshow(photo[0, 0], cmap='gray'); axes[0].set_title('original', fontsize=9)
for ax, (name, k) in zip(axes[1:], kernels.items()):
    kt = torch.tensor(k, dtype=torch.float32)[None, None]     # (1,1,3,3)
    fm = F.conv2d(photo, kt, padding=1)
    ax.imshow(fm[0, 0], cmap='gray'); ax.set_title(name, fontsize=9)
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

같은 이미지, 같은 연산인데 **커널의 숫자만 다르다.** CNN이 하는 일은 이 숫자를
사람이 정하지 않고 **데이터로부터 학습**하는 것이다.

---

# 9. 파라미터 수 — FC와 Conv

In [ ]:
def n_params(m):
    return sum(p.numel() for p in m.parameters())

conv = torch.nn.Conv2d(3, 128, kernel_size=3, padding=1)
fc   = torch.nn.Linear(3*32*32, 128)

print('Conv2d(3→128, k=3) :', f'{n_params(conv):,}',
      ' 공식 3*3*3*128+128 =', f'{3*3*3*128 + 128:,}')
print('Linear(3072→128)   :', f'{n_params(fc):,}',
      ' 공식 3072*128+128  =', f'{3*32*32*128 + 128:,}')
print('\n비율:', round(n_params(fc) / n_params(conv), 1), '배')

콘볼루션의 파라미터 수는 **이미지 크기와 무관하다**. 커널 하나가 모든 위치를
돌아다니며 같은 가중치를 쓰기 때문이다(가중치 공유).

In [ ]:
rows = []
for size in [32, 64, 128, 224]:
    c = torch.nn.Conv2d(3, 128, 3, padding=1)
    f = torch.nn.Linear(3*size*size, 128)
    rows.append({'입력 크기': f'{size}x{size}x3',
                 'Conv2d(3→128,k=3)': f'{n_params(c):,}',
                 'Linear(→128)': f'{n_params(f):,}'})
print(pd.DataFrame(rows).to_string(index=False))

FC는 이미지가 커지면 파라미터가 같이 폭발하지만, Conv는 **그대로**다.

---

# 9.5. 완성 — 콘볼루션으로 실제 분류하기

지금까지 만든 조각으로 **진짜 CNN 하나**를 학습시킨다.
FashionMNIST에서 세 종류(티셔츠 / 바지 / 앵클부츠)만 골라 분류한다.

## 9-5-1. 데이터 준비

In [ ]:
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split

KEEP = [0, 1, 9]
NAMES = [train_raw.classes[i] for i in KEEP]
print('분류할 세 종류:', NAMES)

tf_only = transforms.ToTensor()
tr_full = datasets.FashionMNIST(root='./data', train=True,  download=True, transform=tf_only)
te_full = datasets.FashionMNIST(root='./data', train=False, download=True, transform=tf_only)

y_tr_all = np.array(tr_full.targets); y_te_all = np.array(te_full.targets)
remap = {c: i for i, c in enumerate(KEEP)}

class Picked(Dataset):
    """고른 세 클래스만, 번호를 0/1/2로 다시 매긴다"""
    def __init__(self, base, idx):
        self.base, self.idx = base, idx
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        x, y = self.base[int(self.idx[i])]
        return x, remap[y]

pool = np.concatenate([np.where(y_tr_all == c)[0][:1500] for c in KEEP])
tr_idx, va_idx = train_test_split(pool, test_size=0.2, random_state=42,
                                  stratify=y_tr_all[pool])
te_idx = np.concatenate([np.where(y_te_all == c)[0] for c in KEEP])

train_loader = DataLoader(Picked(tr_full, tr_idx), batch_size=128, shuffle=True)
val_loader   = DataLoader(Picked(tr_full, va_idx), batch_size=256)
test_loader  = DataLoader(Picked(te_full, te_idx), batch_size=256)

print('훈련', len(tr_idx), ' 검증', len(va_idx), ' 테스트', len(te_idx))
xb, yb = next(iter(train_loader))
print('배치 :', tuple(xb.shape), ' 정답 :', tuple(yb.shape))

## 9-5-2. 아주 작은 CNN

In [ ]:
torch.manual_seed(42)
cnn = torch.nn.Sequential(
    torch.nn.Conv2d(1, 8, kernel_size=5, padding=2),   # 커널 8개
    torch.nn.ReLU(),
    torch.nn.MaxPool2d(2),
    torch.nn.AdaptiveAvgPool2d(1),                     # 채널마다 평균 하나 (GAP)
    torch.nn.Flatten(),
    torch.nn.Linear(8, 3),
)

h = torch.zeros(1, 1, 28, 28)
for i, layer in enumerate(cnn):
    h = layer(h)
    print(f'{i} {layer.__class__.__name__:18s} → {tuple(h.shape[1:])}')
print('\n전체 파라미터:', sum(p.numel() for p in cnn.parameters()),
      ' = (5*5*1+1)*8 + (8*3+3)')

> **파라미터 **235개**로 사진을 분류한다**
>
>
> 5장에서 계산했던 완전연결 방식은 $784 \times 128 + 128 = 100{,}480$ 개였다.
> 콘볼루션은 **커널 하나를 모든 위치에서 재사용**하기 때문에 이렇게 작아진다.


## 9-5-3. 학습

In [ ]:
import time
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(cnn.parameters(), lr=0.01)

def evaluate(loader):
    cnn.eval(); correct = n = 0; total = 0.0
    with torch.no_grad():
        for xb, yb in loader:
            out = cnn(xb)
            total += criterion(out, yb).item() * len(yb)
            correct += (out.argmax(1) == yb).sum().item(); n += len(yb)
    return total / n, correct / n

hist = {'train': [], 'val': [], 'acc': []}
t0 = time.time()
for ep in range(8):
    cnn.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss = criterion(cnn(xb), yb)
        loss.backward()
        optimizer.step()
    vl, va = evaluate(val_loader)
    hist['train'].append(loss.item()); hist['val'].append(vl); hist['acc'].append(va)
    print(f'epoch {ep}  val loss {vl:.4f}  val acc {va:.4f}')
print(f'\n학습 시간 {time.time()-t0:.0f}초')

## 9-5-4. 러닝커브와 테스트

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].plot(hist['train'], label='train'); axes[0].plot(hist['val'], label='validation')
axes[0].set_ylabel('cross entropy'); axes[0].legend(fontsize=8)
axes[1].plot(hist['acc'], color='C2'); axes[1].set_ylabel('validation accuracy')
for ax in axes:
    ax.set_xlabel('epoch'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

te_loss, te_acc = evaluate(test_loader)
print(f'테스트 정확도 : {te_acc:.4f}')

## 9-5-5. 모형이 배운 커널을 꺼내 본다

8절에서는 **사람이 설계한** 커널(에지, 블러)을 썼다. 이번에는 **학습으로 얻은** 커널이다.

In [ ]:
kernels = cnn[0].weight.detach().numpy()      # (8, 1, 5, 5)
print('학습된 커널 shape:', kernels.shape)

fig, axes = plt.subplots(1, 8, figsize=(13, 2))
for i, ax in enumerate(axes):
    ax.imshow(kernels[i, 0], cmap='gray')
    ax.set_title(f'kernel {i}', fontsize=8); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# 실제 사진에 통과시켜 무엇에 반응하는지 본다
sample, label = Picked(te_full, te_idx)[0]
with torch.no_grad():
    fmaps = torch.relu(cnn[0](sample.unsqueeze(0)))[0].numpy()

fig, axes = plt.subplots(1, 9, figsize=(14, 2))
axes[0].imshow(sample[0], cmap='gray'); axes[0].set_title(NAMES[label], fontsize=8)
for i in range(8):
    axes[i+1].imshow(fmaps[i], cmap='gray'); axes[i+1].set_title(f'map {i}', fontsize=8)
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

커널마다 **다른 곳이 밝다.** 어떤 것은 세로 윤곽에, 어떤 것은 넓은 면적에 반응한다.
사람이 정해 준 적이 없는데 데이터만 보고 스스로 그렇게 정해졌다.

> **직접 해보기 ③ — 커널 개수를 바꿔 보기**
>
>
> `Conv2d(1, 8, ...)` 의 커널 수를 `2` 와 `32` 로 바꿔 각각 학습시키고
> 테스트 정확도와 파라미터 수를 비교하시오. 커널이 2개면 무엇이 부족한가?

In [ ]:
# ✏️ 직접 채워 보세요
def build_and_train(n_kernels, epochs=8):
    torch.manual_seed(42)
    m = torch.nn.Sequential(...)          # ← 커널 n_kernels개짜리 CNN
    # 위 9-5-3의 학습 루프를 옮겨 오세요
    return m

for k in [2, 8, 32]:
    ...

In [ ]:
# ↓ 정답 코드 (먼저 스스로 해 본 뒤에 실행하세요)
def build_and_train(n_kernels, epochs=8):
    torch.manual_seed(42)
    m = torch.nn.Sequential(
        torch.nn.Conv2d(1, n_kernels, 5, padding=2), torch.nn.ReLU(),
        torch.nn.MaxPool2d(2), torch.nn.AdaptiveAvgPool2d(1),
        torch.nn.Flatten(), torch.nn.Linear(n_kernels, 3))
    opt = torch.optim.Adam(m.parameters(), lr=0.01)
    crit = torch.nn.CrossEntropyLoss()
    for ep in range(epochs):
        m.train()
        for xb, yb in train_loader:
            opt.zero_grad(); crit(m(xb), yb).backward(); opt.step()
    m.eval(); correct = n = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            correct += (m(xb).argmax(1) == yb).sum().item(); n += len(yb)
    return sum(p.numel() for p in m.parameters()), correct / n

rows = []
for k in [2, 8, 32]:
    npar, acc = build_and_train(k)
    rows.append({'커널 수': k, '파라미터': npar, '테스트 정확도': round(acc, 4)})
print(pd.DataFrame(rows).to_string(index=False))

커널이 2개면 볼 수 있는 특성이 두 가지뿐이라 세 종류를 가르기 어렵다.
**커널 수는 "몇 가지 특성을 찾을 것인가"** 를 정하는 하이퍼파라미터다.

---

# 10. 정리

> **이번 주 체크포인트**
>
>
> | 하고 싶은 일 | 코드 | 결과 shape |
> |------|------|------|
> | PIL → 배열 | `np.array(img)` | `(H, W)` 또는 `(H, W, C)` |
> | HWC → CHW | `t.permute(2, 0, 1)` | `(C, H, W)` |
> | CHW → 표시용 | `t.permute(1, 2, 0)` | `(H, W, C)` |
> | 리사이즈 | `transforms.Resize((h, w))` | |
> | 텐서화 + ÷255 | `transforms.ToTensor()` | `(C, H, W)` float32 |
> | 표준화 | `transforms.Normalize(mean, std)` | 그대로 |
> | 배치 한 개 | `next(iter(loader))` | `(N, C, H, W)` |
> | 펴기 | `x.flatten(start_dim=1)` | `(N, C*H*W)` |
> | 콘볼루션 | `F.conv2d(x, k, bias=b, stride=s, padding=p)` | `(N, C_out, OH, OW)` |
> | 출력 크기 | `(N - K + 2*P) // S + 1` | |
> | same 패딩 | `P = (K - 1) // 2` | 크기 유지 |
> | 파라미터 수 | `sum(p.numel() for p in m.parameters())` | |


## 스스로 확인해 보기

아래 결과를 먼저 예상한 뒤 실행해서 대조한다.

In [ ]:
x = torch.randn(8, 3, 28, 28)

c1 = torch.nn.Conv2d(3, 16, kernel_size=5, stride=1, padding=2)
c2 = torch.nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1)

print('입력        :', tuple(x.shape))
print('c1 통과 후  :', tuple(c1(x).shape))
print('c2까지 통과 :', tuple(c2(c1(x)).shape))
print()
print('c1 파라미터 :', sum(p.numel() for p in c1.parameters()))
print('c2 파라미터 :', sum(p.numel() for p in c2.parameters()))
print('펴면 길이   :', c2(c1(x)).flatten(start_dim=1).shape[1])

---

## 다음 실습

[실습 6주차: CNN 구조 만들기](lab06.qmd) —
오늘 만든 콘볼루션 층에 **풀링과 활성화**를 붙여 층을 쌓고, 실제로 학습시킨다.